# 电影情感分析
本实训通过一个广泛使用的电影和电视节目相关的数据集来训练逻辑回归模型，判断某一条影评是好评还是差评。

## 1.数据集介绍


IMDB（Internet Movie Database）数据集是一个广泛使用的电影和电视节目相关的数据集，它来源于同名的在线数据库，IMDB网站，这个网站收集了大量关于电影、电视节目、电视剧集、演员和电影制作团队的信息。
而在本项目中，数据集仅仅包含了用户写的**评论(review)**和对电影的**评价(sentiment)**
其中用户对电影的评价只有两种，分别是**positive**和**negative**

让我们先导入相关依赖

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt


接下来我们看一下前10条数据

In [ ]:
df = pd.read_csv('IMDB_Dataset_.csv')
df['review'] = df['review'].str.replace('<br />', '')

df.head(10)


## 2. 定义函数

### Sigmoid函数

`Sigmoid`函数是在机器学习和深度学习中常用的一种激活函数
**公式**：
$$
\sigma(z)={1\over{1+e^{-z}}}
$$
#### 为什么选用Sigmoid函数作为激活函数

1. *Sigmoid*函数的输出范围是**0到1**。这使得他特别适合用于二分类问题中的概率估计，因为输出值可以直接解释为属于某一类别的概率。

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


### ComputeCost函数

`compute_cost`函数计算损失值函数
计算模型预测值和真实值的差距，在逻辑回归中，一般使用**对数损失（Log Loss）**作为损失函数
**公式**：
$$
J(\theta)=-{1\over m}\sum^{m}_{i=1}[y^{(i)}\log(h_{\theta}(x^{(i)}))+(1-y^{(i)})\log(1-h_{\theta}(x^{(i)}))]
$$
其中$h_{\theta}(x)=\sigma(\theta^{T}x + b)$为模型预测值

In [ ]:
def compute_cost(x: np.ndarray, y: float, weight: np.ndarray):
    # 计算样本总数
    m = x.shape[0]

    # 使用sigmoid函数计算预测概率
    h = sigmoid(np.dot(x, weight))

    # 避免除以0的风险，添加一个非常小的数epsilon
    epsilon = 1e-5

    # 计算代价函数，使用对数似然函数的形式
    cost = -(1 / m) * (np.dot(y, np.log(h + epsilon)) + np.dot((1 - y), np.log(1 - h + epsilon)))

    return cost


### GradientDescent函数

`gradient_descent`是梯度下降函数，用来优化损失值函数

**计算步骤**：

1. 计算预测值：$h=\sigma(\theta^Tx + b)$
2. 计算梯度：$ \nabla J(\theta) = \frac{1}{m} X^T (h - y)$，其中$X$是输入数据矩阵，$y$是标签向量
    - 计算权重的梯度：$gradient_w={1\over m}X^T(h-y)$
    - 计算偏置项的梯度：$gradient_b={1\over m}\sum(h-y)$
3. 更新权重和偏置项
    - $ \theta := \theta - \alpha \times gradient_w $
    - $b := b - \alpha \times gradient_b$
    其中$\alpha$是学习率
4. 记录Cost值：在每次迭代中计算并记录当前代价函数值。

In [ ]:
def gradient_descent(x: np.ndarray, y: float, weights: np.ndarray, learning_rate: float, iterations: int):
    # 计算输入数据集的行数
    m = x.shape[0]
    # 初始化成本历史记录列表
    cost_history = []

    # 使用tqdm显示进度条
    for _ in tqdm(range(iterations)):
        # 预测值的计算
        h = sigmoid(np.dot(x, weights))
        # 计算梯度
        gradient = np.dot(x.T, (h - y)) / m
        # 更新权重
        weights -= learning_rate * gradient
        # 计算当前迭代的cost值
        cost = compute_cost(x, y, weights)
        cost_history.append(cost)

    return weights, cost_history



### Predict函数

使用训练好的全中对输入数据进行预测

**1为正类，0为负类**

In [ ]:
def predict(x: np.ndarray, weights: np.ndarray) -> bool:
    # 计算输入x和权重weights的点积，然后通过激活函数
    h = sigmoid(np.dot(x, weights))
    # 根据sigmoid函数的结果进行预测，如果大于等于0.5，则预测为真，否则预测为假
    return h >= 0.5


1. 清洗数据，将电影评论总得所有换行的HTML标签(`<br />`)删除
2. 初始化`Vectorizer`，将文本数据转换成TF-IDF特征向量
3. 将*sentiment*标签内容转换成0或1（0为'negtive'，1为'positive'）
4. 调用`Sklearn`中的`train_test_split`方法切分数据集，训练集和测试集的比例为**8:2**

由于原数据集较大，运行较慢，因此我们就选择前2000条简单进行演示

In [ ]:
data = df.iloc[:2000]
data['review'] = data['review'].str.replace('<br />', '')

# 特征向量化
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(data['review']).toarray()

# 标签
y = data['sentiment'].apply(lambda x: 1 if x == 'positive' else 0).values

# 划分训练集和测试集
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=43)


+ 初始化参数，包括**权重（weight）**、**学习率（learning_rate）**、**迭代次数（epoch）**
+ 调用`gradient_descent`函数，训练模型，更新权重和偏置值
+ 通过**Sklearn**中的`classification_report`方法，得出Accuracy、Reacll、F1值等指标并利用**seaborn**画出loss值的变化趋势

In [ ]:

weights = np.zeros(x_train.shape[1])
learning_rate = 0.01
epochs = 10

weights, cost_history = gradient_descent(x_train, y_train, weights, learning_rate, epochs)

# 预测测试集
y_pred = predict(x_test, weights)

print(classification_report(y_pred, y_test))

# 绘制损失值变化图
plt.figure(figsize=(6, 4))
sns.lineplot(x=range(1, len(cost_history)+1), y=cost_history)
plt.xlabel("训练轮次", fontsize=12)
plt.ylabel("损失值", fontsize=12)
plt.title("损失函数收敛曲线", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


最后我们对训练好的模型进行测试，我们输入任意一条英文影评，模型就能对其进行判断。

In [ ]:
# ------------------ 一键判片 ------------------
# 输入任意一条英文影评
new_review = ["A thrilling masterpiece with stunning visuals and excellent acting!"]

# 1) 文本→TF-IDF 向量
X_new = vectorizer.transform(new_review).toarray()

# 2) 计算正面概率
prob_pos = sigmoid(np.dot(X_new, weights))[0]

# 3) 用 0.5 做阈值，给出结论
if prob_pos >= 0.5:
    print(f"正面概率 {prob_pos:.2%} → 判断为：好电影")
else:
    print(f"正面概率 {prob_pos:.2%} → 判断为：烂电影")

## 3. 分析

**迭代10次时**:
   - Precision: 类别False为0.93，类别True为0.46。
   - Recall: 类别False为0.61，类别True为0.87。
   - F1-Score: 类别False为0.73，类别True为0.60。
   - 总体准确率：0.68。

